# imports 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# read date from Bronze Layer

In [0]:
df = spark.table('`e-commerce-databricks-project`.bronze_layer.crm_cust_info')
display(df)


# apply Data Transformations for silver Table

In [0]:
df.display()

## Renaming columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

## Remove Records with Missing Customer ID

In [0]:
df = df.filter(F.col('customer_id').isNotNull())
df = df.withColumn('customer_id', F.col('customer_id').cast(StringType()))

## Triming String columns

In [0]:
for colName, colType in df.dtypes:
    if colType == 'string':
        df = df.withColumn(colName, F.trim(F.col(colName)))
df.display()

## Data Normalization for gender and marital_status

In [0]:
df = df.withColumn('marital_status',
    F.when(F.lower(F.col('marital_status')) == 's', 'Single')
    .when(F.lower(F.col('marital_status')) == 'm', 'Married')
    .otherwise('Unknown'))

df = df.withColumn('gender',
    F.when(F.upper(F.col('gender')) == 'M', 'Male')
    .when(F.upper(F.col('gender')) == 'F', 'Female')
    .otherwise('Unknown'))


## Final Look

In [0]:
df.display()

# write data into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('`e-commerce-databricks-project`.silver_layer.customer_info')